# wrangle / portfolio / profile

**Intent**
- Estimates return, volatility and shortfall from `portfolio_profile_risky_share_pct` and `portfolio_profile_withdrawal_rate_pct`, and must:
  - Give `portfolio_profile_return_pct` and `portfolio_profile_volatility_pct`, in percent a year.
  - Give `portfolio_profile_shortfall_probability_pct`, the chance of running out within 30 years.
  - Simulate the shortfall from that same return and volatility, so they cannot disagree.
  - Treat the withdrawal rate as a percent of initial wealth, constant in real terms.
  - Let any mix and withdrawal be compared on risk, return and shortfall.
  - Hold return and volatility to 0.05 points, and shortfall to 10 points.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is thirty share and withdrawal combinations, `validation` is twenty further, and `backtest` is twenty across the range.
- `backtest` is held back for acceptance.
- `golden` is shares from 0 to 100 percent with withdrawal rates from 2 to 6, then a request with an input missing.

**Status**
- Two assets and their correlation are recovered from synthetic points.
- The mix return is a line and its variance a quadratic in the risky share.
- Shortfall is a logistic curve fitted to a fixed seed simulation, and the returns are real.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add `query_postgres` to `collect_run` and load the real series in place of the synthetic ones.
  - Draw returns by stationary block bootstrap from the `portfolio/return` history, re-centred on its forward base.
  - Draw inflation from `household/costs` jointly with returns, and use survival from `portfolio/withdrawal` instead of 30 years.
  - Report the shortfall probability, the expected shortfall and the fifth percentile terminal wealth.
  - Try a spending cut rule, since being able to cut helps ruin more than any asset choice.
  - End in one table per candidate risky share, with sensitivities and the worst drawdown.
  - Decide first the spending, horizon, tolerance for a fall, other income and bequest, which the model cannot.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/portfolio/profile")


GOLDEN = "portfolio_profile_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from math import sqrt

import numpy as np

from mlmodels import collect_run

YEARS = 30
PATHS = 2000


def mix(share):
    weight = share / 100.0
    variance = (weight * 15.0) ** 2 + ((1.0 - weight) * 5.0) ** 2 + 2.0 * 0.2 * weight * (1.0 - weight) * 15.0 * 5.0
    return 2.0 + 4.0 * weight, sqrt(variance)


def shortfall(share, rate):
    mean, volatility = mix(share)
    returns = np.random.default_rng(0).normal(mean / 100.0, volatility / 100.0, (PATHS, YEARS))
    wealth = np.full(PATHS, 100.0)
    for year in range(YEARS):
        wealth = (wealth - rate) * (1.0 + returns[:, year])
        wealth = np.where(wealth > 0.0, wealth, 0.0)
    return float(np.mean(wealth <= 0.0) * 100.0)


def labelled(shares, rates):
    pairs = [(share, rate) for share in shares for rate in rates]
    return {
        ws.inputs[0]: [share for share, _ in pairs],
        ws.inputs[1]: [rate for _, rate in pairs],
        ws.outputs[0]: [round(mix(share)[0], 6) for share, _ in pairs],
        ws.outputs[1]: [round(mix(share)[1], 6) for share, _ in pairs],
        ws.outputs[2]: [round(shortfall(share, rate), 4) for share, rate in pairs],
    }


collect_run(
    ws,
    1000,
    label="Thirty synthetic share and withdrawal combinations",
    name="portfolio_profile_training",
    source="builder",
    build=lambda: labelled([10.0, 30.0, 50.0, 70.0, 90.0], [2.0, 3.0, 4.0, 5.0, 6.0, 7.0]),
)
collect_run(ws, 1000, label="Twenty further synthetic combinations", name="portfolio_profile_validation", source="builder", build=lambda: labelled([20.0, 40.0, 60.0, 80.0], [2.5, 3.5, 4.5, 5.5, 6.5]))
collect_run(
    ws,
    1000,
    label="Twenty synthetic combinations across the range",
    name="portfolio_profile_backtest",
    source="builder",
    build=lambda: labelled([15.0, 35.0, 55.0, 75.0, 95.0], [3.25, 4.25, 5.25, 6.25]),
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "portfolio_profile_training"), validation=ws.run(1000, "portfolio_profile_validation"), backtest=ws.run(1000, "portfolio_profile_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
from scipy.optimize import curve_fit

weights = ws.training[ws.inputs[0]].to_numpy() / 100.0
rates = ws.training[ws.inputs[1]].to_numpy()
means = ws.training[ws.outputs[0]].to_numpy()
volatilities = ws.training[ws.outputs[1]].to_numpy()
return_slope, return_intercept = np.polyfit(weights, means, 1)
variance_square, variance_linear, variance_constant = np.polyfit(weights, volatilities**2, 2)
fitted = curve_fit(
    lambda data, slope, m0, m1, m2: 100.0 / (1.0 + np.exp(np.minimum(-slope * (data[0] - (m0 + m1 * data[1] - m2 * data[2])), 50.0))),
    (rates, means, volatilities),
    ws.training[ws.outputs[2]].to_numpy(),
    p0=[1.0, 3.0, 1.0, 0.05],
    maxfev=20000,
)[0]
ws.params = {
    "return_intercept": round(float(return_intercept), 4),
    "return_slope": round(float(return_slope), 4),
    "variance_square": round(float(variance_square), 4),
    "variance_linear": round(float(variance_linear), 4),
    "variance_constant": round(float(variance_constant), 4),
    "slope": round(float(fitted[0]), 4),
    "m0": round(float(fitted[1]), 4),
    "m1": round(float(fitted[2]), 4),
    "m2": round(float(fitted[3]), 4),
}
print("fitted", ws.params)
assert ws.params["return_intercept"] == 2.0 and ws.params["return_slope"] == 4.0, "the fit did not recover the asset returns"
assert ws.params["slope"] > 0.0, "the shortfall probability does not rise with the withdrawal rate"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

RETURN_INTERCEPT, RETURN_SLOPE = ws.params["return_intercept"], ws.params["return_slope"]
VARIANCE_SQUARE, VARIANCE_LINEAR, VARIANCE_CONSTANT = ws.params["variance_square"], ws.params["variance_linear"], ws.params["variance_constant"]
SLOPE, M0, M1, M2 = ws.params["slope"], ws.params["m0"], ws.params["m1"], ws.params["m2"]


def profile(share, rate):
    from math import exp, sqrt

    weight = share / 100.0
    mean = RETURN_INTERCEPT + RETURN_SLOPE * weight
    volatility = sqrt(max(VARIANCE_SQUARE * weight**2 + VARIANCE_LINEAR * weight + VARIANCE_CONSTANT, 0.0))
    shortfall = 100.0 / (1.0 + exp(min(-SLOPE * (rate - (M0 + M1 * mean - M2 * volatility)), 50.0)))
    return mean, volatility, shortfall


define_model(
    ws,
    model=1000,
    flavour="pyfunc",
    source=profile,
    label="Two asset blend with a logistic shortfall curve",
    constants={
        "RETURN_INTERCEPT": ws.params["return_intercept"],
        "RETURN_SLOPE": ws.params["return_slope"],
        "VARIANCE_SQUARE": ws.params["variance_square"],
        "VARIANCE_LINEAR": ws.params["variance_linear"],
        "VARIANCE_CONSTANT": ws.params["variance_constant"],
        "SLOPE": ws.params["slope"],
        "M0": ws.params["m0"],
        "M1": ws.params["m1"],
        "M2": ws.params["m2"],
    },
)

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models

LIMITS = dict(zip(ws.outputs, [0.05, 0.05, 10.0], strict=True))


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        frame = getattr(ws, split)
        predicted = evaluate_model(ws, 1000, split)
        checks.extend(
            {
                "split": split,
                "gate": split == "backtest",
                "output": output,
                "name": "rmse",
                "value": evaluate_metrics(frame[output], predicted[output])["rmse"],
                "limit": LIMITS[output],
                "direction": "max",
            }
            for output in ws.outputs
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

shares = [0.0, 25.0, 50.0, 75.0, 100.0]
rates = [2.0, 3.0, 4.0, 5.0, 6.0]
golden = {ws.inputs[0]: [*shares, None], ws.inputs[1]: [*rates, rates[0]], "expect": ["ok"] * len(shares) + ["error"]}
package_model(ws, 1000, golden, target={"kind": "local"})